# Assignment 4: Coffee sales window-function puzzles

Complete **all six puzzles** by replacing the `____` markers in their SQL cells. Run each query and any provided plot, answer the short written prompts, and submit your completed notebook through Canvas. The plotting code is already written.

**Grading: 30 points total.**

| Puzzle | Query and plot, if provided | Written response | Total |
|---|---:|---:|---:|
| 1. Running total | 5 | — | 5 |
| 2. Rolling average | 4 | 1 | 5 |
| 3. Previous sale day | 3 | 2 | 5 |
| 4. Monthly leader | 4 | 1 | 5 |
| 5. `NTILE` buckets | 3 | 2 | 5 |
| 6. Calendar-day frame | 3 | 2 | 5 |
| **Total** | **22** | **8** | **30** |

Use the class workday to begin the puzzles; finish any remaining work independently. The notebook uses [Yaroslav Isaienkov's coffee-vending-machine sales data](https://www.kaggle.com/datasets/ihelon/coffee-sales), **version 21**, licensed CC0. We use `index_1.csv` and only complete months from **March–December 2024**, so everyone works with the same rows. The source calls the payment column `money` but does not specify a currency; we call it `amount` and do not attach a currency symbol. Each row is a purchase recorded in the source file.

The first run downloads the public Kaggle file and may install missing Python packages. It needs internet access; later runs can use KaggleHub's cache. Public Kaggle datasets generally do not need a login. If a login is requested, use your own account or ask the instructor for the CSV. Never put credentials in a notebook you submit.

## Setup: one transaction table and three summaries

Run the next two cells without editing them. The embedded DuckDB database lets you practice SQL window functions without connecting to a PostgreSQL server. The window syntax used in this assignment also works in PostgreSQL.

- `coffee_sales`: one row per purchase. `drink` is the product category; `amount` is the recorded payment.
- `daily_sales`: one row per **recorded** date, with a count of drinks and total amount.
- `daily_drink_sales`: one row per recorded date and drink.
- `monthly_drink_sales`: one row per month and drink.

The daily summaries omit dates with no transactions. A frame of seven **rows** therefore means seven recorded dates, which may differ from seven consecutive calendar days.

In [ ]:
import importlib.util
import subprocess
import sys

needed = []
for module, package in [
    ("duckdb", "duckdb>=1.1,<2"),
    ("kagglehub", "kagglehub>=0.3,<1"),
    ("matplotlib", "matplotlib>=3.5"),
]:
    if importlib.util.find_spec(module) is None:
        needed.append(package)
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *needed])

import duckdb
import kagglehub
import matplotlib.pyplot as plt

csv_path = kagglehub.dataset_download(
    "ihelon/coffee-sales/versions/21", path="index_1.csv"
)
print(f"Downloaded or found cached CSV: {csv_path}")

In [ ]:
con = duckdb.connect()
con.execute("""
    CREATE TABLE coffee_sales AS
    SELECT CAST(date AS DATE) AS sale_date,
           CAST(datetime AS TIMESTAMP) AS sale_time,
           cash_type AS payment_type,
           CAST(money AS DECIMAL(10, 2)) AS amount,
           coffee_name AS drink
    FROM read_csv_auto(?)
    WHERE CAST(date AS DATE) BETWEEN DATE '2024-03-01' AND DATE '2024-12-31'
""", [csv_path])

con.execute("""
    CREATE VIEW daily_sales AS
    SELECT sale_date, COUNT(*) AS drinks_sold, SUM(amount) AS revenue
    FROM coffee_sales
    GROUP BY sale_date
""")
con.execute("""
    CREATE VIEW daily_drink_sales AS
    SELECT sale_date, drink, COUNT(*) AS drinks_sold
    FROM coffee_sales
    GROUP BY sale_date, drink
""")
con.execute("""
    CREATE VIEW monthly_drink_sales AS
    SELECT CAST(DATE_TRUNC('month', sale_date) AS DATE) AS sale_month,
           drink, COUNT(*) AS drinks_sold
    FROM coffee_sales
    GROUP BY sale_month, drink
""")

def run_and_preview(sql, shown=6):
    result = con.execute(sql)
    columns = [column[0] for column in result.description]
    rows = result.fetchall()
    print(f"{len(rows):,} rows | " + " | ".join(columns))
    for row in rows[:shown]:
        print(row)
    if len(rows) > shown:
        print("...")
    return rows

counts = con.execute("""
    SELECT (SELECT COUNT(*) FROM coffee_sales) AS transactions,
           (SELECT COUNT(*) FROM daily_sales) AS recorded_dates,
           (SELECT COUNT(DISTINCT drink) FROM coffee_sales) AS drinks
""").fetchone()
print(f"Transactions: {counts[0]:,}; recorded dates: {counts[1]}; drinks: {counts[2]}")
assert counts == (2693, 302, 8), "Check that Kaggle version 21/index_1.csv loaded."
run_and_preview("SELECT * FROM daily_sales ORDER BY sale_date LIMIT 5")

## Puzzle 1 — How many drinks have sold so far? (5 points)

`daily_sales` already has one row per recorded date. Complete the aggregate, window order, and frame so `drinks_to_date` is a **running total** from March 1 through the current row. Keep all 302 daily rows. Check that the last running total equals the 2,693 transactions loaded above.

In [ ]:
puzzle_1_sql = """
SELECT sale_date, drinks_sold,
       SUM(____) OVER (
           ORDER BY ____
           ROWS BETWEEN ____ AND ____
       ) AS drinks_to_date
FROM daily_sales
ORDER BY sale_date;
"""
puzzle_1_rows = run_and_preview(puzzle_1_sql)
print("Last row:", puzzle_1_rows[-1])

In [ ]:
# Ready-made plot for Puzzle 1. Run after the SQL cell works.
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot([row[0] for row in puzzle_1_rows],
        [row[2] for row in puzzle_1_rows], color="#176b87")
ax.set(title="Cumulative drinks sold, March–December 2024",
       xlabel="Recorded date", ylabel="Drinks sold to date")
ax.grid(alpha=0.25)
fig.autofmt_xdate()
plt.show()

## Puzzle 2 — Smooth the daily ups and downs (5 points)

Complete the aggregate and window so the output is the average number of drinks sold on the **current row and six previous recorded dates**. The first few rows use the observations available so far. Round the average to one decimal place. Run the plot to compare the raw daily count with its rolling average.

Remember: four dates in this period have no transaction row, so this is a seven-*recorded*-date average, not always a seven-calendar-day average.

In [ ]:
puzzle_2_sql = """
SELECT sale_date, drinks_sold,
       ROUND(AVG(____) OVER (
           ORDER BY ____
           ROWS BETWEEN ____ PRECEDING AND CURRENT ROW
       ), 1) AS avg_7_recorded_dates
FROM daily_sales
ORDER BY sale_date;
"""
puzzle_2_rows = run_and_preview(puzzle_2_sql)

In [ ]:
# Ready-made plot for Puzzle 2. Run after the SQL cell works.
dates = [row[0] for row in puzzle_2_rows]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(dates, [row[1] for row in puzzle_2_rows],
        color="#9cb8c1", linewidth=1, label="Daily drinks")
ax.plot(dates, [float(row[2]) for row in puzzle_2_rows],
        color="#b45309", linewidth=2, label="Seven recorded dates, average")
ax.set(title="Daily drink sales and a rolling average",
       xlabel="Recorded date", ylabel="Drinks sold")
ax.legend()
ax.grid(alpha=0.25)
fig.autofmt_xdate()
plt.show()

**Observation (1 point; one sentence):** Where does the rolling average make a pattern easier to see? Replace this sentence with your answer.

## Puzzle 3 — Compare a drink with its previous recorded sale day (5 points)

In `daily_drink_sales`, one row represents one drink on one date when it sold at least once. Complete **both** `LAG` calls: find the previous recorded date and its count *for the same drink*. The outer query computes the change. Run the prepared plot to see October's changes for Latte.

In [ ]:
puzzle_3_sql = """
WITH previous AS (
    SELECT sale_date, drink, drinks_sold,
           LAG(____) OVER (
               PARTITION BY ____ ORDER BY sale_date
           ) AS previous_date,
           LAG(____) OVER (
               PARTITION BY ____ ORDER BY sale_date
           ) AS previous_drinks_sold
    FROM daily_drink_sales
    WHERE drink IN ('Latte', 'Espresso')
)
SELECT drink, sale_date, drinks_sold,
       previous_date, previous_drinks_sold,
       drinks_sold - previous_drinks_sold AS change_in_drinks
FROM previous
ORDER BY drink, sale_date;
"""
puzzle_3_rows = run_and_preview(puzzle_3_sql, shown=10)

In [ ]:
# Ready-made plot for Puzzle 3. Run after the SQL cell works.
from datetime import date

latte_october = [
    row for row in puzzle_3_rows
    if row[0] == "Latte" and date(2024, 10, 1) <= row[1] <= date(2024, 10, 31)
]
changes = [row[5] for row in latte_october]
colors = ["#176b87" if change >= 0 else "#b45309" for change in changes]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar([row[1] for row in latte_october], changes, color=colors)
ax.axhline(0, color="#334155", linewidth=0.8)
ax.set(title="Latte: change from the previous recorded sale date, October 2024",
       xlabel="Sale date", ylabel="Change in drinks sold")
ax.grid(axis="y", alpha=0.25)
fig.autofmt_xdate()
plt.show()

**Interpretation (2 points; two sentences):** Why is the first row for each drink's change `NULL`? What does a negative bar mean, and is its comparison always with the previous *calendar* day? Replace this text with your answer.

## Puzzle 4 — Which drink led each month? (5 points)

The view `monthly_drink_sales` already counts purchases by month and drink. Complete the ranking function so it restarts each month and ranks drinks by count, highest first. Use `drink` as the alphabetical tie-breaker. Keep only the winner. The result should have one row for each of the ten months.

In [ ]:
puzzle_4_sql = """
WITH ranked AS (
    SELECT sale_month, drink, drinks_sold,
           ____() OVER (
               PARTITION BY ____
               ORDER BY ____ DESC, ____
           ) AS position
    FROM monthly_drink_sales
)
SELECT sale_month, drink, drinks_sold
FROM ranked
WHERE position = ____
ORDER BY sale_month;
"""
puzzle_4_rows = run_and_preview(puzzle_4_sql, shown=10)

In [ ]:
# Ready-made plot for Puzzle 4. Run after the SQL cell works.
months = [row[0].strftime("%b") for row in puzzle_4_rows]
counts = [row[2] for row in puzzle_4_rows]
drinks = [row[1] for row in puzzle_4_rows]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(months, counts, color="#176b87")
for bar, drink in zip(bars, drinks):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            drink, ha="center", va="bottom", rotation=90, fontsize=8)
ax.set(title="Top-selling drink each month, March–December 2024",
       xlabel="Month", ylabel="Purchases")
ax.set_ylim(0, max(counts) * 1.45)
plt.tight_layout()
plt.show()

**Observation (1 point; one sentence):** Did the same drink lead every month? Replace this sentence with your answer based on the query output.

## Puzzle 5 — Place each drink in a monthly sales bucket (5 points)

`NTILE(4)` places each month's **eight drink categories** into four buckets of two rows each. Complete the window so bucket 1 contains the two highest-selling drinks in each month; break ties alphabetically by `drink`. The prepared code prints October's eight rows after running your query.

**Interpretation:** Identify the two drinks in October's bucket 1. In October, Americano and Cappuccino each sold 44 drinks but appear in different buckets. Why can `NTILE` split tied values?

In [ ]:
puzzle_5_sql = """
SELECT sale_month, drink, drinks_sold,
       NTILE(____) OVER (
           PARTITION BY ____
           ORDER BY ____ DESC, ____
       ) AS sales_bucket
FROM monthly_drink_sales
ORDER BY sale_month, sales_bucket, drinks_sold DESC, drink;
"""
puzzle_5_rows = run_and_preview(puzzle_5_sql)
print("October buckets:")
for row in puzzle_5_rows:
    if row[0].isoformat() == "2024-10-01":
        print(row)

**Answer (2 points; two sentences):** Replace this text with October's bucket-1 drinks and your explanation of the tied 44-drink rows.

## Puzzle 6 — Recorded rows versus calendar days (5 points)

The lecture distinguished `ROWS` from `RANGE`. Complete the second aggregate and its **calendar-time frame** so it sums purchases from the current date and the six preceding calendar days. The first window is already a seven-recorded-date sum. Compare the results around May 6, when the source has dates with no sales rows. Which question does each column answer?

In [ ]:
puzzle_6_sql = """
SELECT sale_date, drinks_sold,
       SUM(drinks_sold) OVER (
           ORDER BY sale_date
           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
       ) AS seven_recorded_dates,
       SUM(____) OVER (
           ORDER BY ____
           ____ BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
       ) AS seven_calendar_days
FROM daily_sales
ORDER BY sale_date;
"""
puzzle_6_rows = run_and_preview(puzzle_6_sql)
print("Around the dates with no sales rows:")
for row in puzzle_6_rows:
    if '2024-05-02' <= row[0].isoformat() <= '2024-05-08':
        print(row)

**Explanation (2 points):** Replace this sentence with your comparison of the two seven-day interpretations.

## Before submitting

- Complete and run **all six puzzles**, their provided plots, and the short observations and explanations.
- Leave the SQL queries and their outputs visible. If a query fails, fix it before submitting rather than replacing its output with a screenshot.
- Check that window `ORDER BY` controls the calculation, while the final `ORDER BY` controls result display.

**References:** [PostgreSQL window-function lecture](https://JacobDowns.github.io/CSCI340/lectures/week-11-window-functions.html); [DuckDB window functions](https://duckdb.org/docs/stable/sql/functions/window_functions.html); [KaggleHub downloads](https://github.com/Kaggle/kagglehub#download-dataset). Dataset: [Coffee Sales by Yaroslav Isaienkov](https://www.kaggle.com/datasets/ihelon/coffee-sales), version 21, CC0 Public Domain.